# HR Candidate Profile Parser (Colab Edition — Free LLM)

**Goal:** Parse a resume (a real PDF you upload, or a sample text) into a clean, structured
JSON object using LangChain.

**LLM provider:** [Google Gemini]

**Why `PydanticOutputParser`:** Our schema has nested lists of objects (`education`,
`experience`), not just flat fields. `PydanticOutputParser` lets us define real nested models
(`Education`, `Experience`, `CandidateProfile`), which:
- Generates precise format instructions for the LLM automatically.
- Validates and type-casts the parsed result (e.g. `year` is guaranteed to be an `int`).
- Raises a clear validation error if the model returns something malformed.

**Pipeline:**
- Upload your own CV as a PDF and extract its raw text (or use the built-in sample text).
- Define the target schema as Pydantic models.
- Build a prompt that instructs the LLM to follow that schema.
- Call the LLM and parse its response into a validated object.
- Display and save the final JSON.

## Setup

In [ ]:
%pip install -q -U langchain langchain-google-genai pydantic pypdf

In [ ]:
import os
import json

## Input: Upload Your CV (PDF) or Use the Sample Text

Run the upload cell to pick your CV PDF from your computer. Its text will be extracted
automatically. If you skip the upload, the built-in sample text is used instead.

In [ ]:
sample_resume_text = """John Smith – john.smith@email.com
Education: B.Sc. Computer Science, MIT, 2020
Skills: Python, Machine Learning, Data Analysis
Experience:
- Software Engineer at Google (2020–2023)
- Data Scientist at OpenAI (2023–Present)
"""

In [ ]:
from pypdf import PdfReader


def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract raw text from every page of a PDF file."""
    reader = PdfReader(pdf_path)
    pages_text = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages_text).strip()

In [ ]:
resume_text = sample_resume_text  # default fallback

try:
    from google.colab import files

    print("Upload your CV as a PDF (or cancel/close this dialog to use the sample text instead):")
    uploaded = files.upload()

    if uploaded:
        pdf_filename = next(iter(uploaded))
        resume_text = extract_text_from_pdf(pdf_filename)
        print(f"\nExtracted text from '{pdf_filename}':\n")
except Exception as e:
    print(f"Upload skipped or unavailable ({e}). Using sample text instead.")

print(resume_text)

## Define the Schema with Pydantic

In [ ]:
from typing import List, Optional
from pydantic import BaseModel, Field


class Education(BaseModel):
    degree: str = Field(description="The degree obtained, e.g. 'B.Sc. Computer Science'")
    institution: str = Field(description="The institution's name, e.g. 'MIT'")
    year: Optional[int] = Field(default=None, description="The graduation year, e.g. 2020, if mentioned")


class Experience(BaseModel):
    role: str = Field(description="The job title, e.g. 'Software Engineer'")
    company: str = Field(description="The company name, e.g. 'Google'")
    years: str = Field(description="The employment period, e.g. '2020-2023' or '2023-Present'")


class CandidateProfile(BaseModel):
    full_name: str = Field(description="The candidate's full name")
    email: Optional[str] = Field(default=None, description="The candidate's email address, if present")
    education: List[Education] = Field(default_factory=list, description="List of education entries")
    skills: List[str] = Field(default_factory=list, description="List of skills")
    experience: List[Experience] = Field(default_factory=list, description="List of work experience entries")

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=CandidateProfile)
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

## Build the Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    """You are an HR assistant that extracts structured candidate data from resumes.

Read the resume text below and extract the requested fields exactly as instructed.
Do not invent information that is not present in the text. If a field is not present,
leave it empty or null as appropriate.

Resume text:
---
{resume_text}
---

{format_instructions}"""
)

prompt = prompt_template.format_messages(
    resume_text=resume_text,
    format_instructions=format_instructions,
)

print(prompt[0].content)

## Call the LLM (Gemini — Free)

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
response = llm.invoke(prompt)
print(response.content)

## Parse and Validate the Response

In [ ]:
def extract_text(response) -> str:
    """Extract plain text from a LangChain AIMessage response,
    handling both plain-string content and list-of-blocks content."""
    if isinstance(response.content, str):
        return response.content
    return "".join(
        block.get("text", "") for block in response.content if isinstance(block, dict)
    )

In [ ]:
candidate_profile: CandidateProfile = output_parser.parse(extract_text(response))
print(json.dumps(candidate_profile.model_dump(), indent=2))

## Wrap It in a Reusable Function

In [ ]:
def parse_candidate_profile(text: str) -> CandidateProfile:
    messages = prompt_template.format_messages(
        resume_text=text,
        format_instructions=format_instructions,
    )
    llm_response = llm.invoke(messages)
    return output_parser.parse(extract_text(llm_response))

In [ ]:
result = parse_candidate_profile(resume_text)
print(json.dumps(result.model_dump(), indent=2))

## Save the Output to a JSON File

In [ ]:
output_path = "candidate_profile.json"

with open(output_path, "w", encoding="utf-8") as f:
    f.write(result.model_dump_json(indent=2))

print(f"Saved parsed profile to {output_path}")